# Cuevas v1: cuevas de la Luna y Marte frente a las cuevas y tubos de lava de la Tierra

Capa para el uso **«cuevas y hábitat»** del plan A + D. cuevas_v0 ya tenía la Tierra (cuevas documentadas + entradas de
OpenStreetMap en las regiones candidatas). v1 añade **la Luna y Marte** y lo cruza con la topografía, como en hielo_v0.

| | Dato | Qué da |
|---|---|---|
| Luna | LROC, catálogo de pozos lunares (Wagner y Robinson 2021) | pozos (*pits*) en mares, tierras altas y fundidos de impacto |
| Marte | Mars Global Cave Candidate Catalog (Cushing 2015/2017, USGS/PDS) | más de 1000 candidatos a entrada de cueva (tragaluces, pozos atípicos, fracturas) |
| Tierra | OpenStreetMap `natural=cave_entrance` + GLiM + cuevas documentadas | entradas de cueva y posibles tubos de lava por región |

**Clase de cueva** (sitios y regiones): `tubo_lava` (pozo en lava / entrada en roca volcánica), `cueva` (otro tipo) o ninguna.
**Distancia de cuevas** (sitio con cueva a ≤ `RADIO_CUEVA_KM` frente a región): 0 misma clase · 0,5 otra clase · 1 región sin cuevas.
Los sitios sin cuevas cerca quedan sin distancia (la capa no aplica), como los sitios sin hielo.

**Requisitos**: topografia_sitios_v2 (versión nueva) y cuevas_v0 (caché de Overpass y `cuevas_osm.csv`).

## 1. Configuración

In [13]:
import os, re, json, glob, zipfile, shutil, datetime, unicodedata, warnings, time
import numpy as np, pandas as pd, xarray as xr, requests, geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Transformer
from shapely.geometry import shape, Point
from shapely import affinity
from shapely.ops import transform as transformar
from IPython.display import display
from google.colab import drive
warnings.filterwarnings("ignore", message=".*geographic CRS.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

drive.mount("/content/drive")
for raiz in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
    if os.path.isdir(raiz):
        break
BASE = f"{raiz}/Hackathon Nasa"
VERSION = "cuevas_v1"
RES, WEB = f"{BASE}/04_resultados/{VERSION}", f"{BASE}/06_web/{VERSION}"
BRUTOS = f"{BASE}/03_datos/brutos/cuevas"
CACHE_OSM = f"{BASE}/03_datos/cache_api/overpass"          # la misma caché que cuevas_v0
PROCESADOS = f"{BASE}/03_datos/procesados"
TMP = "/content/tmp/cuevas"
for d in (RES, WEB, BRUTOS, CACHE_OSM, TMP):
    os.makedirs(d, exist_ok=True)

R_TIERRA, R_LUNA, R_MARTE = 6371.0, 1737.4, 3389.5     # km
RADIO_CUEVA_KM = {"luna": 50, "marte": 50}   # un sitio «tiene cueva» si hay un pozo o candidato a esta distancia (alcance de un rover)
RADIO_ANALOGO_KM, MARGEN_KM = 10, 25         # análogos como círculos de 10 km; cuevas a ≤ 25 km de la región (como cuevas_v0)
UMBRAL_TOPO, UMBRAL_CUEVAS = 0.2, 0.0        # análogo cueva + terreno: misma clase de cueva y topografía ≤ 0,2
D_CLASE = {("tubo_lava", "tubo_lava"): 0.0, ("cueva", "cueva"): 0.0, ("tubo_lava", "cueva"): 0.5, ("cueva", "tubo_lava"): 0.5}
TOPO_TODAS = f"{BASE}/04_resultados/topografia_sitios_v2/similitud_sitio_tierra_todas.csv"
OSM_V0 = f"{BASE}/04_resultados/cuevas_v0/cuevas_osm.csv"

FIGURAS = f"{RES}/figuras"; os.makedirs(FIGURAS, exist_ok=True)
_mostrar = getattr(plt, "_show_original", plt.show); plt._show_original = _mostrar
_n_fig = [0]
def _mostrar_y_guardar(*a, **k):
    fig = plt.gcf()
    if fig.axes:
        _n_fig[0] += 1
        titulo = next((ax.get_title() for ax in fig.axes if ax.get_title()), "figura")
        s = re.sub(r"[^a-zA-Z0-9]+", "_", unicodedata.normalize("NFKD", titulo).encode("ascii", "ignore").decode()).strip("_").lower()[:50]
        fig.savefig(f"{FIGURAS}/fig_{_n_fig[0]:02d}_{s}.png", dpi=80, bbox_inches="tight")
    _mostrar(*a, **k)
plt.show = _mostrar_y_guardar

SESION = requests.Session()
SESION.headers["User-Agent"] = "Mozilla/5.0 (Hackathon NASA Space Apps; Colab)"

def descargar(urls, destino, minimo_bytes=1000, es_zip=False):
    valido = lambda p: os.path.exists(p) and os.path.getsize(p) >= minimo_bytes and (not es_zip or zipfile.is_zipfile(p))
    if valido(destino):
        return destino
    for url in urls:
        try:
            tmp = f"{TMP}/{os.path.basename(destino)}.part"
            with SESION.get(url, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for trozo in r.iter_content(1 << 22):
                        f.write(trozo)
            if valido(tmp):
                shutil.copy2(tmp, destino); os.remove(tmp)
                print(f"   ✔ {os.path.basename(destino)} ({os.path.getsize(destino) / 1e6:.2f} MB) desde {url[:70]}")
                return destino
            print(f"   ⚠ {url[:80]}: el archivo no es válido ({os.path.getsize(tmp) if os.path.exists(tmp) else 0:,} bytes)")
        except Exception as e:
            print(f"   ⚠ {url[:80]}: {e!r}"[:200])
    return None

def slug(t):
    return re.sub(r"[^a-z0-9]+", "-", unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower()).strip("-")[:40]

def km_entre(lat1, lon1, lat2, lon2, R):
    la1, lo1, la2, lo2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

entrada = json.load(open(f"{BASE}/06_web/manifest.json", encoding="utf-8"))
sitios = gpd.read_file(f"{BASE}/06_web/sitios_v0/sitios.geojson")
sitios["lon"] = sitios.geometry.centroid.x; sitios["lat"] = sitios.geometry.centroid.y
print(sitios.groupby(["cuerpo", "tipo"]).size())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuerpo  tipo                     
luna    aterrizaje_planificado        1
        aterrizaje_robotico          22
        aterrizaje_tripulado          6
        region_aterrizaje_artemis     9
marte   aterrizaje_planificado        1
        aterrizaje_robotico          11
        exploration_zone             47
dtype: int64


## 2. Descargas
- **Luna**: CSV del catálogo de pozos de LROC (y, si falla, el shapefile en dominio ±180°).
- **Marte**: zip del catálogo de candidatos a cueva (Astropedia). Si no baja, el notebook busca los archivos de datos en el
  directorio del PDS de USGS. Si nada funciona: descarga `mars_cave_catalog.zip` desde
  https://astrogeology.usgs.gov/search/map/mars_global_cave_candidate_catalog_v1_cushing y súbelo a `03_datos/brutos/cuevas/`.

In [14]:
RUTA_PITS = "LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/EXTRAS/SHAPEFILE/LUNAR_PIT_LOCATIONS/"
PITS_URLS = [f"https://pds.lroc.im-ldi.com/data/{RUTA_PITS}LUNAR_PIT_LOCATIONS.CSV",
             f"https://pds.lroc.asu.edu/data/{RUTA_PITS}LUNAR_PIT_LOCATIONS.CSV",
             "https://pds.mcp.nasa.gov/data/store/img/lunar_reconnaissance_orbiter/pds4/lroc/lro-l-lroc-5-rdr/LROLRC_2001/EXTRAS/SHAPEFILE/LUNAR_PIT_LOCATIONS/LUNAR_PIT_LOCATIONS.CSV"]
PITS_SHP = [f"https://pds.lroc.im-ldi.com/data/{RUTA_PITS}LUNAR_PIT_LOCATIONS_180.ZIP"]

ruta = descargar(PITS_URLS, f"{BRUTOS}/LUNAR_PIT_LOCATIONS.CSV", minimo_bytes=5_000)
if ruta:
    pits = pd.read_csv(ruta, encoding="latin-1")
else:
    z = descargar(PITS_SHP, f"{BRUTOS}/LUNAR_PIT_LOCATIONS_180.ZIP", minimo_bytes=5_000, es_zip=True)
    assert z, f"Sin catálogo de pozos lunares: súbelo a {BRUTOS}/ desde {PITS_URLS[0]}"
    pits = pd.DataFrame(gpd.read_file(f"zip://{z}").drop(columns="geometry"))
pits.columns = [c.strip() for c in pits.columns]
print(f"Pozos lunares: {len(pits)} · columnas: {list(pits.columns)}")

MGC_URLS = ["https://astrogeology.usgs.gov/ckan/dataset/c2960f77-ee20-4cc6-ac85-abbecc0fc7f6/resource/ccd54fb2-9a50-46bf-8971-80c241bb2f40/download/mars_cave_catalog.zip",
            "https://astropedia.astrogeology.usgs.gov/download/Mars/MarsCaveCatalog/mars_cave_catalog.zip",
            "http://astropedia.astrogeology.usgs.gov/download/Mars/MarsCaveCatalog/mars_cave_catalog.zip",
            "https://asc-astropedia.s3.us-west-2.amazonaws.com/Mars/MarsCaveCatalog/mars_cave_catalog.zip"]
DIR_PDS_MGC = "https://pdsimage2.wr.usgs.gov/downloads/mars_mro.odyssey_multi_cavecatalog_cushing_2016/"
z = descargar(MGC_URLS, f"{BRUTOS}/mars_cave_catalog.zip", minimo_bytes=5_000, es_zip=True)
DIR_MGC = f"{TMP}/mgc"
if z:
    with zipfile.ZipFile(z) as zz:
        zz.extractall(DIR_MGC)
else:   # plan B: recorrer el directorio del PDS (dos niveles) y bajar las tablas
    os.makedirs(DIR_MGC, exist_ok=True)
    pendientes, vistos = [DIR_PDS_MGC], set()
    for _ in range(3):
        nuevos = []
        for u in pendientes:
            try:
                html = SESION.get(u, timeout=120).text
            except Exception as e:
                print(f"   ⚠ {u}: {e!r}"[:150]); continue
            for h in re.findall(r'href="([^"?#]+)"', html):
                v = requests.compat.urljoin(u, h)
                if not v.startswith(DIR_PDS_MGC) or v in vistos:
                    continue
                vistos.add(v)
                if v.endswith("/"):
                    nuevos.append(v)
                elif re.search(r"\.(csv|tab|txt|xlsx?|xml|dbf|shp|shx|prj)$", v, re.I) and "/document" not in v.lower():
                    descargar([v], f"{DIR_MGC}/{os.path.basename(v)}", minimo_bytes=50)
        pendientes = nuevos
print("Archivos del catálogo de Marte:", [os.path.relpath(p, DIR_MGC) for p in glob.glob(f"{DIR_MGC}/**/*", recursive=True) if os.path.isfile(p)][:40])

Pozos lunares: 278 · columnas: ['Name', 'Terrain', 'Type', 'Host Feature', 'Latitude', 'Longitude_180', 'Longitude_360', 'Funnel Max Diam', 'Funnel Min Diam', 'Inner Max Diam', 'Inner Min Diam', 'Azimuth', 'Depth', 'Depth_Num', 'Depth1', 'Depth2', 'Depth3', 'Image 1', 'Image 1 Line', 'Image 1 Sample', 'Inc 1', 'Em 1', 'Time 1', 'Res 1', 'Image 2', 'Inc 2', 'Em 2', 'Time 2', 'Res 2', 'Image 3', 'Inc 3', 'Em 3', 'Time 3', 'Res 3', 'Stereo IDs', 'DTM', 'Other Useful Images', 'Warnings', 'Description', 'Entrance Ramp?', 'Overhangs?', 'Nearby Pits?', 'Nearby Fractures?', 'Nearby Depressions?', 'In Dome?', 'Beside Dome?', 'In Depression?', 'Colinear Pits?', 'Colinear depressions?', 'Colinear Fractures?']
Archivos del catálogo de Marte: ['MGC3_README.xml', 'mars_mro.odyssey_multi_cavecatalog_cushing_2016.xml', 'MGC3_README.pdf', 'miscellaneous/collection_misc_inventory.csv', 'miscellaneous/collection_misc_inventory.xml', 'miscellaneous/Mars_Cave_Catalog_shapefile.csv', 'miscellaneous/Mars_Cav

### 2.1 Lectura del catálogo de Marte
El formato interno del zip no está documentado en la web, así que se buscan todas las tablas (CSV, TAB del PDS con su etiqueta
XML, Excel, shapefile) y se queda la que tenga columnas de latitud y longitud y más filas. **Revisa la salida**: si las columnas
detectadas no son las correctas, cámbialas a mano en `COL_LAT`, `COL_LON` y `COL_TIPO`.

In [15]:
COL_LAT, COL_LON, COL_TIPO = None, None, None   # rellenar a mano solo si la detección automática falla

def nombres_de_etiqueta(xml):
    try:
        t = open(xml, encoding="utf-8", errors="ignore").read()
    except Exception:
        return None
    campos = re.findall(r"<Field_(?:Delimited|Character)>.*?<name>(.*?)</name>", t, re.S)
    return [c.strip() for c in campos] or None

def leer_tabla(p):
    ext = p.lower().rsplit(".", 1)[-1]
    if ext in ("xlsx", "xls"):
        return pd.read_excel(p)
    if ext in ("shp", "dbf"):
        return pd.DataFrame(gpd.read_file(p).drop(columns="geometry", errors="ignore"))
    nombres = None
    for xml in (p[: -len(ext)] + "xml", p[: -len(ext)] + "XML"):
        if os.path.exists(xml):
            nombres = nombres_de_etiqueta(xml)
    df = pd.read_csv(p, sep=None, engine="python", encoding="latin-1", on_bad_lines="skip")
    if nombres and len(nombres) == df.shape[1] and not any(re.search("lat", str(c), re.I) for c in df.columns):   # sin cabecera: nombres de la etiqueta XML
        df = pd.read_csv(p, sep=None, engine="python", encoding="latin-1", header=None, names=nombres, on_bad_lines="skip")
    return df

def columna(df, patrones, evitar=()):
    for pat in patrones:
        for c in df.columns:
            n = str(c).lower()
            if re.search(pat, n) and not any(re.search(e, n) for e in evitar):
                return c
    return None

mejor = None
for p in glob.glob(f"{DIR_MGC}/**/*", recursive=True):
    if not re.search(r"\.(csv|tab|txt|xlsx?|shp)$", p, re.I) or re.search(r"readme|inventory|collection|errata", p, re.I):
        continue
    try:
        df = leer_tabla(p)
    except Exception as e:
        print(f"   ⚠ {os.path.basename(p)}: {e!r}"[:150]); continue
    la = COL_LAT or columna(df, [r"^lat", r"lat"], evitar=[r"err|unc|sigma"])
    lo = COL_LON or columna(df, [r"^e?lon", r"lon", r"long"], evitar=[r"err|unc|sigma"])
    print(f"   {os.path.relpath(p, DIR_MGC)}: {df.shape} · lat={la} · lon={lo}")
    if la is not None and lo is not None and (mejor is None or len(df) > len(mejor[0])):
        mejor = (df, la, lo, p)
assert mejor, f"No se encontró una tabla con latitud y longitud en el catálogo de Marte ({DIR_MGC})"
mgc, la, lo, ARCHIVO_MGC = mejor
mgc = mgc.rename(columns={la: "lat", lo: "lon"})
mgc["lat"] = pd.to_numeric(mgc.lat, errors="coerce"); mgc["lon"] = pd.to_numeric(mgc.lon, errors="coerce")
mgc = mgc.dropna(subset=["lat", "lon"])
mgc["lon"] = ((mgc.lon + 180) % 360) - 180
COL_TIPO = COL_TIPO or columna(mgc, [r"type", r"class", r"categ", r"morph", r"feature"])
print(f"\nMarte: {len(mgc)} candidatos desde {os.path.basename(ARCHIVO_MGC)} · columna de tipo: {COL_TIPO}")
print("Columnas:", list(mgc.columns))
if COL_TIPO:
    print(mgc[COL_TIPO].astype(str).value_counts().head(20))
display(mgc.head())
for p in glob.glob(f"{DIR_MGC}/**/*", recursive=True):   # leyenda de los códigos de tipo (README del zip)
    if re.search(r"readme|description", os.path.basename(p), re.I) and p.lower().endswith((".txt", ".md")):
        t = open(p, encoding="latin-1").read()
        lineas = [l for l in t.splitlines() if re.search(r"\b(sky|srp|apc|crk|pin|flr|pit|irr|rim|lat|end|pol|kst|con|smrp)\b", l, re.I)]
        print(f"\n{os.path.basename(p)}: líneas con códigos de tipo"); print("\n".join(lineas[:40]))

   miscellaneous/Mars_Cave_Catalog_shapefile.csv: (1062, 9) · lat=latitude · lon=longitude
   data/Mars_Cave_Catalog.csv: (1062, 8) · lat=latitude:string · lon=longitude:string

Marte: 1062 candidatos desde Mars_Cave_Catalog_shapefile.csv · columna de tipo: TypeCode:string
Columnas: ['lon', 'lat', 'Label:string', 'TypeCode:string', 'Priority:string', 'APC_Diameter:string', 'APC_Depth:string', 'Comment:string', 'Feature:string']
TypeCode:string
sky     354
srp     218
APC     132
crk      87
pin      60
flr      42
pit      37
irr      36
rim      30
lat      27
end      21
pol       8
kst       7
con       2
smrp      1
Name: count, dtype: int64


,lon,lat,Label:string,TypeCode:string,Priority:string,APC_Diameter:string,APC_Depth:string,Comment:string,Feature:string
0,-119.54150,-0.94678,APC001,APC,1,25,32.0,APC -- tiny ~25m,point
1,-122.30396,-12.47266,APC003,APC,1,30,47.0,APC 40m small deep nochain--big smrp?,point
2,-119.11328,-5.68445,APC005,APC,1,30,17.0,APC deep pair,point
3,-119.05786,-1.61377,APC006,APC,1,40,38.0,APC small deep?,point
4,-118.70068,-1.68555,APC007,APC,1,40,28.0,"APC, deep? ~40m",point


## 3. Clase de cueva en la Luna y Marte
- **Luna**: pozo en **mar** → `tubo_lava` (tragaluz posible a un tubo de lava); en fundido de impacto o tierras altas → `cueva`.
- **Marte**: el tipo del catálogo con «skylight», «lava», «tube» o pozo atípico (APC) → `tubo_lava`; fracturas, karst y el resto →
  `cueva`. Si el catálogo no trae tipo, se usa la geología del sitio (geologia_v0): basalto → `tubo_lava`.

In [ ]:
c_lat = columna(pits, [r"^lat"]); c_lon = columna(pits, [r"^lon", r"lon"]); c_ter = columna(pits, [r"terrain"])
c_nom = columna(pits, [r"^name"]); c_diam = columna(pits, [r"funmaxdiam", r"diam"]); c_prof = columna(pits, [r"^depth$", r"depth"])
luna_p = pd.DataFrame({"nombre": pits[c_nom] if c_nom else None, "lat": pd.to_numeric(pits[c_lat], errors="coerce"),
                       "lon": pd.to_numeric(pits[c_lon], errors="coerce"),
                       "terreno": pits[c_ter].astype(str).str.strip().str.lower() if c_ter else "",
                       "diametro_m": pd.to_numeric(pits[c_diam], errors="coerce") if c_diam else np.nan,
                       "profundidad_m": pd.to_numeric(pits[c_prof], errors="coerce") if c_prof else np.nan}).dropna(subset=["lat", "lon"])
luna_p["lon"] = ((luna_p.lon + 180) % 360) - 180
luna_p["clase"] = np.where(luna_p.terreno.str.contains("mare"), "tubo_lava", "cueva")
luna_p["cuerpo"] = "luna"
print("Luna:", luna_p.terreno.value_counts().to_dict(), "→", luna_p.clase.value_counts().to_dict())

tipo = mgc[COL_TIPO].astype(str) if COL_TIPO else pd.Series("", index=mgc.index)
PAT_TUBO = re.compile(r"sky|lava|tube|apc|atypical", re.I)
texto = mgc[[c for c in mgc.columns if mgc[c].dtype == object and c != COL_TIPO]]   # el nombre o código es una columna de texto
c_id = columna(texto, [r"^id", r"name", r"^cc", r"id"]) if len(texto.columns) else None
marte_p = pd.DataFrame({"nombre": mgc[c_id].astype(str) if c_id else ("MGC-" + mgc.index.astype(str)),
                        "lat": mgc.lat, "lon": mgc.lon, "tipo_catalogo": tipo})
marte_p["clase"] = np.where(tipo.str.contains(PAT_TUBO), "tubo_lava", "cueva") if COL_TIPO else None
marte_p["cuerpo"] = "marte"
print("Marte:", marte_p.clase.value_counts(dropna=False).to_dict())

## 4. Cuevas cerca de cada sitio
Distancia (km, sobre la esfera del cuerpo) al pozo o candidato más cercano, número a ≤ `RADIO_CUEVA_KM` y clase. Si hay de las dos
clases cerca, manda `tubo_lava` (es el caso que más interesa como refugio).

In [ ]:
GEO_V0 = f"{BASE}/04_resultados/geologia_v0/geologia_topografia_sitios.csv"
clase_geo = dict(pd.read_csv(GEO_V0)[["sitio_id", "clase_dominante"]].values) if os.path.exists(GEO_V0) else {}

filas = []
for r in sitios.itertuples():
    cat, R = (luna_p, R_LUNA) if r.cuerpo == "luna" else (marte_p, R_MARTE)
    d = km_entre(r.lat, r.lon, cat.lat.values, cat.lon.values, R)
    j = int(np.argmin(d)); cerca = cat[d <= RADIO_CUEVA_KM[r.cuerpo]]
    clases = set(cerca.clase.dropna())
    if r.cuerpo == "marte" and not COL_TIPO and len(cerca):
        clases = {"tubo_lava" if clase_geo.get(r.id) == "basalto" else "cueva"}
    clase = "tubo_lava" if "tubo_lava" in clases else ("cueva" if clases else None)
    filas.append({"id": r.id, "nombre": r.nombre, "cuerpo": r.cuerpo, "tipo": r.tipo, "lat": round(r.lat, 3), "lon": round(r.lon, 3),
                  "cueva_mas_cercana": cat.nombre.iloc[j], "km_cueva_mas_cercana": round(float(d[j]), 1),
                  "clase_mas_cercana": cat.clase.iloc[j], "n_cuevas_radio": len(cerca), "clase_cueva": clase})
cue_sit = pd.DataFrame(filas)
print(cue_sit.groupby("cuerpo").apply(lambda t: f"{t.clase_cueva.notna().sum()} de {len(t)} con cueva a ≤ 50 km", include_groups=False).to_string())
display(cue_sit.sort_values(["cuerpo", "km_cueva_mas_cercana"]).head(30))

## 5. Tierra: cuevas por región
Las mismas regiones que en topografía, geología e hielo (candidatas + análogos conocidos), más las cuevas documentadas de cuevas_v0
y las cuevas de hielo del Erebus. Entradas de OSM dentro de la región o a ≤ 25 km; las candidatas reutilizan la caché de cuevas_v0
y los análogos hacen una consulta nueva a Overpass (se guarda en la misma caché).

In [ ]:
CUEVAS_REF = pd.DataFrame([
    ["La Corona (Cueva de los Verdes / Túnel de la Atlántida), Lanzarote", 29.16, -13.44, "luna", "tubo_lava", "https://blogs.esa.int/caves/?p=2369"],
    ["Valentine Cave, Lava Beds National Monument, California", 41.72, -121.51, "marte", "tubo_lava", "https://www.nasa.gov/feature/ames/braille"],
    ["Tubos de lava del Mauna Loa (HI-SEAS), Hawái", 19.60, -155.48, "luna", "tubo_lava",
     "https://www.hawaiinewsnow.com/2021/04/14/prepare-life-mars-scientists-are-descending-into-mauna-loas-lava-tubes/"],
    ["Surtshellir–Stefánshellir (Hallmundarhraun), Islandia", 64.78, -20.72, "luna", "tubo_lava", "https://indico.stfc.ac.uk/event/1058/contributions/6721/"],
    ["Cuevas de hielo fumarólicas del Erebus (Warren Cave)", -77.5167, 167.1647, "marte", "cueva", "https://doi.org/10.3389/fmicb.2015.00179"],
], columns=["nombre", "lat", "lon", "perfil", "clase", "fuente"])

def circulo_grados(lon, lat, radio_km):
    rlat = radio_km / 111.32
    return affinity.scale(Point(lon, lat).buffer(rlat, 64), xfact=1 / max(np.cos(np.radians(lat)), 0.05), yfact=1)

filas = []
for cuerpo in ["luna", "marte"]:
    info = entrada["cuerpos"][cuerpo]
    for ft in json.load(open(f"{BASE}/06_web/{info['carpeta']}/regiones.geojson", encoding="utf-8"))["features"]:
        p = ft["properties"]
        filas.append({"region_id": p["id"], "perfil": cuerpo, "tipo_tierra": "region_candidata", "region": p.get("region"), "geom": shape(ft["geometry"])})
    for ft in json.load(open(f"{BASE}/06_web/{info['carpeta']}/analogos.geojson", encoding="utf-8"))["features"]:
        p = ft["properties"]
        ok = (p.get("tipo") in ("analogo_luna", "impacto")) if cuerpo == "luna" else (p.get("verificacion") == "verificada")
        if ok and ft.get("geometry"):
            lon, lat = ft["geometry"]["coordinates"][:2]
            filas.append({"region_id": f"analogo-{cuerpo}-{slug(p['nombre'])}", "perfil": cuerpo, "tipo_tierra": "analogo_conocido",
                          "region": p["nombre"], "geom": circulo_grados(lon, lat, RADIO_ANALOGO_KM)})
ids = {f["region_id"] for f in filas}
for c in CUEVAS_REF.itertuples():
    rid = f"analogo-{c.perfil}-{slug(c.nombre)}"
    if rid not in ids:
        filas.append({"region_id": rid, "perfil": c.perfil, "tipo_tierra": "cueva_documentada", "region": c.nombre,
                      "geom": circulo_grados(c.lon, c.lat, RADIO_ANALOGO_KM)})
tierra = gpd.GeoDataFrame(filas, geometry="geom", crs="EPSG:4326")
print(tierra.groupby(["perfil", "tipo_tierra"]).size())

SERVIDORES = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
def consultar_overpass(clave, consulta):
    ruta = f"{CACHE_OSM}/{clave}.json"
    if os.path.exists(ruta):
        return json.load(open(ruta, encoding="utf-8"))
    for intento in range(4):
        try:
            r = requests.post(SERVIDORES[intento % 2], data={"data": consulta}, timeout=180,
                              headers={"User-Agent": "space-apps-2026-analogos (educational)"})
            if r.status_code == 200:
                datos = r.json(); json.dump(datos, open(ruta, "w", encoding="utf-8"), ensure_ascii=False); time.sleep(2)
                return datos
        except Exception as e:
            print(f"   ⚠ Overpass {clave}: {e!r}"[:120])
        time.sleep(10 * (intento + 1))
    print(f"   ⚠ Overpass sin respuesta para {clave}: se cuenta como sin dato"); return None

def caja_con_margen(geom, km):
    oeste, sur, este, norte = geom.bounds
    dlat = km / 111.32; dlon = km / (111.32 * max(np.cos(np.radians((sur + norte) / 2)), 0.1))
    return max(sur - dlat, -90), max(oeste - dlon, -180), min(norte + dlat, 90), min(este + dlon, 180)

puntos, sin_dato = [], set()
for r in tierra.itertuples():
    s, o, n, e = caja_con_margen(r.geom, MARGEN_KM)
    clave = f"region_{r.region_id}" if r.tipo_tierra == "region_candidata" else f"analogo_{slug(r.region_id)}"
    q = f'[out:json][timeout:120];nwr["natural"="cave_entrance"]({s:.4f},{o:.4f},{n:.4f},{e:.4f});out center tags;'
    datos = consultar_overpass(clave, q)
    if datos is None:
        sin_dato.add(r.region_id); continue
    for el in datos.get("elements", []):
        la_, lo_ = (el.get("lat"), el.get("lon")) if el["type"] == "node" else (el.get("center", {}).get("lat"), el.get("center", {}).get("lon"))
        if la_ is not None:
            t = el.get("tags", {})
            puntos.append({"osm_id": f"{el['type']}/{el['id']}", "lat": la_, "lon": lo_, "nombre": t.get("name"), "etiquetas": json.dumps(t, ensure_ascii=False)})
osm = pd.DataFrame(puntos).drop_duplicates("osm_id").reset_index(drop=True)
print(f"Entradas de cueva distintas en las cajas: {len(osm):,} · regiones sin respuesta de Overpass: {len(sin_dato)}")

glim = xr.open_dataset(f"{PROCESADOS}/glim_005.nc")
LAT_G, LON_G = glim.lat.values, glim.lon.values
PASO = 180 / len(LAT_G)
volc = sum(glim.frac.sel(clase=c).values.astype("uint16") for c in ["vb", "vi", "va", "py"]).clip(0, 100)
def volcanica(la_, lo_):
    i = np.clip(np.round((np.asarray(la_) - LAT_G[0]) / PASO).astype(int), 0, len(LAT_G) - 1)
    j = np.clip(np.round((np.asarray(lo_) - LON_G[0]) / PASO).astype(int), 0, len(LON_G) - 1)
    return volc[i, j]
PALABRAS_LAVA = re.compile(r"lava|tubo|tunnel|túnel|hellir|volcan|basalt|jameo", re.I)
osm["posible_tubo_lava"] = (volcanica(osm.lat, osm.lon) >= 50) | osm.etiquetas.str.contains(PALABRAS_LAVA) | osm.nombre.fillna("").str.contains(PALABRAS_LAVA)

filas = []
for r in tierra.itertuples():
    c = r.geom.centroid
    a_local = Transformer.from_crs("EPSG:4326", f"+proj=aeqd +lat_0={c.y} +lon_0={c.x} +units=km", always_xy=True).transform
    g = transformar(a_local, r.geom)
    x, y = a_local(osm.lon.values, osm.lat.values)
    dentro = np.array([g.distance(Point(px, py)) <= MARGEN_KM for px, py in zip(x, y)], dtype=bool) if len(osm) else np.array([], bool)
    o = osm[dentro]
    xr_, yr_ = a_local(CUEVAS_REF.lon.values, CUEVAS_REF.lat.values)
    doc = CUEVAS_REF[[g.distance(Point(px, py)) <= MARGEN_KM for px, py in zip(xr_, yr_)]]
    n_tubos = int(o.posible_tubo_lava.sum()) + int((doc.clase == "tubo_lava").sum())
    n = len(o) + len(doc)
    clase = None if (r.region_id in sin_dato and not len(doc)) else ("tubo_lava" if n_tubos else ("cueva" if n else "sin_cuevas"))
    filas.append({"region_id": r.region_id, "entradas_osm": len(o), "posibles_tubos_lava": n_tubos, "cuevas_documentadas": "; ".join(doc.nombre),
                  "clase_cueva_tierra": clase, "ejemplos": "; ".join(o.nombre.dropna().unique()[:3])})
tierra = tierra.merge(pd.DataFrame(filas), on="region_id")
display(tierra.drop(columns="geom").sort_values(["posibles_tubos_lava", "entradas_osm"], ascending=False).head(25))
print(tierra.groupby("perfil").clase_cueva_tierra.value_counts(dropna=False))

## 6. Comparación: cuevas, y cuevas + terreno
Para cada sitio **con cueva** cerca y cada región de la Tierra: `d_cuevas` según la clase (0 / 0,5 / 1). Se cruza con la
distancia topográfica (los dos perfiles) y, para cada sitio, se busca la región con la misma clase de cueva y el terreno más
parecido. **Análogo cueva + terreno** = misma clase y topografía ≤ 0,2.

In [ ]:
S = cue_sit[cue_sit.clase_cueva.notna()][["id", "nombre", "cuerpo", "tipo", "clase_cueva", "km_cueva_mas_cercana", "n_cuevas_radio"]]
T = tierra[["region_id", "region", "perfil", "tipo_tierra", "clase_cueva_tierra", "entradas_osm", "posibles_tubos_lava", "cuevas_documentadas"]]
pares = S.merge(T, how="cross")
pares["d_cuevas"] = [D_CLASE.get((a, b), 1.0) if b is not None and not (isinstance(b, float) and np.isnan(b)) else np.nan
                     for a, b in zip(pares.clase_cueva, pares.clase_cueva_tierra)]
if os.path.exists(TOPO_TODAS):
    topo = pd.read_csv(TOPO_TODAS)[["sitio_id", "region_id", "distancia"]].rename(columns={"sitio_id": "id", "distancia": "d_topo"})
    pares = pares.merge(topo, on=["id", "region_id"], how="left")
else:
    print("⚠ Falta similitud_sitio_tierra_todas.csv (topografia_sitios_v2): sin combinación con el terreno"); pares["d_topo"] = np.nan
pares["d_cuevas_topo"] = ((pares.d_cuevas + pares.d_topo) / 2).round(3)
pares.to_csv(f"{RES}/similitud_cuevas.csv", index=False)

misma = pares[pares.d_cuevas <= UMBRAL_CUEVAS]
mejor_ct = misma.dropna(subset=["d_topo"]).sort_values("d_topo").drop_duplicates("id").set_index("id")   # no first(): mezclaría filas
res = cue_sit.merge(mejor_ct[["region", "perfil", "clase_cueva_tierra", "posibles_tubos_lava", "cuevas_documentadas", "d_cuevas", "d_topo"]].add_suffix("_ct"),
                    left_on="id", right_index=True, how="left")
res["analogo_cueva_terreno"] = np.where(res.d_topo_ct <= UMBRAL_TOPO, "Sí", np.where(res.clase_cueva.notna(), "No", "—"))
res.to_csv(f"{RES}/cuevas_sitios.csv", index=False)
tierra.drop(columns="geom").to_csv(f"{RES}/cuevas_tierra.csv", index=False)
pd.concat([luna_p, marte_p]).to_csv(f"{RES}/cuevas_luna_marte.csv", index=False)

cols = ["nombre", "tipo", "clase_cueva", "cueva_mas_cercana", "km_cueva_mas_cercana", "n_cuevas_radio", "region_ct", "d_topo_ct",
        "cuevas_documentadas_ct", "analogo_cueva_terreno"]
for cuerpo in ["luna", "marte"]:
    d = res[(res.cuerpo == cuerpo) & res.clase_cueva.notna()].sort_values("d_topo_ct")
    print(f"{cuerpo.capitalize()}: {len(d)} sitios con cueva a ≤ {RADIO_CUEVA_KM[cuerpo]} km"); display(d[cols])
print(res.groupby("cuerpo").analogo_cueva_terreno.value_counts())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for ax, (cuerpo, cat) in zip(axs, (("luna", luna_p), ("marte", marte_p))):
    for clase, color in (("tubo_lava", "black"), ("cueva", "tab:gray")):
        c = cat[cat.clase == clase]; ax.scatter(c.lon, c.lat, s=4, c=color, label=f"{clase} ({len(c)})")
    s = res[res.cuerpo == cuerpo]
    ax.scatter(s.lon, s.lat, s=40, marker="^", c=np.where(s.analogo_cueva_terreno == "Sí", "tab:green",
               np.where(s.clase_cueva.notna(), "tab:orange", "tab:red")), edgecolors="k", label="sitios (verde = análogo cueva + terreno)")
    ax.set_xlim(-180, 180); ax.set_ylim(-90, 90); ax.grid(alpha=0.3); ax.legend(fontsize=7, loc="lower left")
    ax.set_title(f"{cuerpo.capitalize()}: pozos / candidatos a cueva y sitios")
plt.show()

## 7. Capa para la web y registro

In [ ]:
def limpio(v):
    if isinstance(v, (float, np.floating)): return None if not np.isfinite(v) else round(float(v), 3)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.bool_): return bool(v)
    return v

top5 = misma.dropna(subset=["d_topo"]).sort_values("d_topo").groupby("id").head(5)
salida = {"version": VERSION,
          "unidades": {"km_cueva_mas_cercana": "km sobre la esfera del cuerpo", "clase_cueva": "tubo_lava | cueva | null (sin cueva a ≤ 50 km)",
                       "d_cuevas": "0 misma clase · 0,5 otra clase · 1 región sin cuevas", "d_topo": "distancia topográfica (topografia_sitios_v2)"},
          "umbrales": {"radio_cueva_km": RADIO_CUEVA_KM, "topografia": UMBRAL_TOPO},
          "sitios": {r["id"]: {**{k: limpio(v) for k, v in r.items() if k != "id"},
                               "top5_tierra": [{k: limpio(v) for k, v in p.items() if k in ("region_id", "region", "perfil", "clase_cueva_tierra",
                                                                                            "posibles_tubos_lava", "cuevas_documentadas", "d_cuevas", "d_topo")}
                                               for p in top5[top5.id == r["id"]].to_dict("records")]}
                     for r in res.to_dict("records")},
          "tierra": {r["region_id"]: {k: limpio(v) for k, v in r.items() if k not in ("region_id", "geom")}
                     for r in tierra.drop(columns="geom").to_dict("records")},
          "nota": "unir con 06_web/sitios_v0/sitios.geojson por id; region_id como en topografia_v2"}
json.dump(salida, open(f"{WEB}/cuevas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, allow_nan=False)
feats = [{"type": "Feature", "geometry": {"type": "Point", "coordinates": [round(r.lon, 4), round(r.lat, 4)]},
          "properties": {"cuerpo": r.cuerpo, "nombre": limpio(r.nombre), "clase": r.clase}} for r in pd.concat([luna_p, marte_p]).itertuples()]
json.dump({"type": "FeatureCollection", "features": feats, "metadata": {"crs": "longitud/latitud planetocéntricas de cada cuerpo, −180 a 180"}},
          open(f"{WEB}/cuevas_luna_marte.geojson", "w", encoding="utf-8"), ensure_ascii=False)
entrada = json.load(open(f"{BASE}/06_web/manifest.json", encoding="utf-8"))
entrada.setdefault("capas", {})["cuevas_sitios"] = {"version": VERSION, "carpeta": VERSION, "archivo": f"{VERSION}/cuevas.json",
                                                    "puntos": f"{VERSION}/cuevas_luna_marte.geojson"}
json.dump(entrada, open(f"{BASE}/06_web/manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

con = lambda c: res[(res.cuerpo == c) & res.clase_cueva.notna()]
registro = {"version": VERSION, "fecha": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="minutes"),
            "fuentes": {"luna_pozos": PITS_URLS[0], "marte_cuevas": MGC_URLS[0], "marte_archivo": os.path.basename(ARCHIVO_MGC),
                        "tierra": "OpenStreetMap natural=cave_entrance (Overpass) + GLiM + cuevas documentadas"},
            "parametros": {"radio_cueva_km": RADIO_CUEVA_KM, "margen_tierra_km": MARGEN_KM, "radio_analogo_km": RADIO_ANALOGO_KM,
                           "umbral_topo": UMBRAL_TOPO, "columna_tipo_marte": COL_TIPO},
            "metricas": {"pozos_luna": len(luna_p), "candidatos_marte": len(marte_p),
                         "luna_sitios_con_cueva": len(con("luna")), "marte_sitios_con_cueva": len(con("marte")),
                         "luna_analogo_cueva_terreno": int((con("luna").analogo_cueva_terreno == "Sí").sum()),
                         "marte_analogo_cueva_terreno": int((con("marte").analogo_cueva_terreno == "Sí").sum()),
                         "regiones_tierra_con_tubo_lava": int((tierra.clase_cueva_tierra == "tubo_lava").sum()),
                         "regiones_tierra_con_cueva": int((tierra.clase_cueva_tierra == "cueva").sum()),
                         "regiones_tierra_sin_dato": int(tierra.clase_cueva_tierra.isna().sum())},
            "limitaciones": ["un pozo o candidato no confirma una cueva: muchos pueden ser huecos poco profundos",
                             "la clase tubo_lava/cueva se deduce del terreno (Luna) o del tipo del catálogo (Marte)",
                             "OpenStreetMap cubre de forma desigual: una región sin cuevas puede estar sin mapear",
                             "la distancia de cuevas es categórica (0 / 0,5 / 1); no compara tamaño ni profundidad"]}
json.dump(registro, open(f"{RES}/registro_{VERSION}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=str)
print("✅ Exportado:", json.dumps(registro["metricas"], ensure_ascii=False, indent=1))